# Algebraic Tables of Angular-Momentum Coefficients

Symbolic generators for the algebraic tables in Varshalovich, Moskalev &
Khersonskii, *Quantum Theory of Angular Momentum*:

| Section | Tables | Object | Generator |
|---|---|---|---|
| 8.12 | 8.1–8.10 | Clebsch–Gordan $\langle a\alpha, b\beta \mid c\gamma\rangle$, $b=\tfrac12\ldots5$ | `gen_8_12_cg_tables` |
| 9.11 | 9.1–9.8 | $6j$ $\begin{Bmatrix}a&b&c\\d&e&f\end{Bmatrix}$, $d=\tfrac12\ldots4$ | `gen_9_11_6j_tables` |
| 10.11 | 10.1–10.12 | $9j$ with third row $(\alpha,\beta,\gamma)$ | `gen_10_11_9j_tables` |

Each produces an **exact** entry — as rendered LaTeX in the book's own notation
and as a SymPy expression — for any cell, including values beyond the ranges
the book prints.

**Method.** Each table fixes one small momentum and leaves the rest symbolic, so
the relevant Racah sum has only a few terms. Changing to integer-valued
variables ($p=a{+}\alpha,\,q=a{-}\alpha$ for §8.12; the triangle deficits
$u=s{-}2a$ etc. for §9.11) makes every factorial argument an explicit integer
offset, so all factorials collapse to rising/falling factorials and the algebra
stays rational — no `simplify()` anywhere. The $9j$ of §10.11 is expanded over
three $6j$'s and reuses the §9.11 engine.

Every generator is verified against SymPy's own `CG` / `wigner_6j` / `wigner_9j`
(see the last section).

## Setup

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / "scripts" / "gen_8_12_cg_tables.py").exists():
        ROOT = cand
        sys.path.insert(0, str(cand / "scripts"))
        break
else:
    raise RuntimeError("could not locate the VMK scripts/ directory from " + str(here))

from sympy import Rational
from IPython.display import display, Math, Markdown, HTML

import gen_8_12_cg_tables as cg8
import gen_9_11_6j_tables as sixj9
import gen_10_11_9j_tables as ninej10

try:
    import ipywidgets as W
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

print(f"repo root : {ROOT}")
print(f"widgets   : {'yes' if HAS_WIDGETS else 'no  (dropdowns disabled; edit-and-rerun cells still work)'}")

repo root : /Users/mccssnrw/Documents/Projects/VMK
widgets   : yes


Widgets are **optional**. Without them every section still works through a
plain *edit the values, re-run the cell* form.

To enable the dropdowns, run `%pip install ipywidgets` in a cell below and
restart the kernel. That installs into whichever environment this kernel uses.

## Helpers

In [2]:
def _vals(x):
    """The index values a table runs over: x, x-1, ..., -x."""
    x = Rational(x)
    return [x - i for i in range(int(2 * x) + 1)]


def _fmt(x):
    return str(Rational(x))


def _shift(base, k):
    """'c', 'c+1/2', 'b-1' ..."""
    k = Rational(k)
    return base if k == 0 else f"{base}{'+' if k > 0 else '-'}{_fmt(abs(k))}"


def _show(heading, latex_str, expr, show_source):
    display(Markdown(f"**{heading}**"))
    display(Math(latex_str))
    if expr is not None:
        display(Markdown(f"SymPy: `{expr}`"))
    if show_source:
        display(Markdown("LaTeX source:\n```latex\n" + latex_str + "\n```"))


def _grid(rows, cols, cell, row_label, col_label, corner):
    """Render a table of LaTeX cells as HTML (MathJax typesets the $...$)."""
    html = ["<style>.vmk td,.vmk th{border:1px solid #999;padding:4px 8px;"
            "font-size:90%;text-align:center}.vmk{border-collapse:collapse}</style>",
            "<table class='vmk'><tr><th>" + corner + "</th>"]
    html += [f"<th>{col_label(c)}</th>" for c in cols]
    html.append("</tr>")
    for r in rows:
        html.append(f"<tr><th>{row_label(r)}</th>")
        for c in cols:
            html.append("<td>$" + cell(r, c) + "$</td>")
        html.append("</tr>")
    html.append("</table>")
    return HTML("".join(html))

---
## §8.12 — Clebsch–Gordan coefficients

$$\langle a\,\alpha,\; b\,\beta \mid c\,\gamma\rangle, \qquad c = a+k,\quad \gamma=\alpha+\beta$$

with $a,\alpha$ symbolic. Pick the fixed $b$, the **row** $k = c-a$ and the
**column** $\beta$; both run over $b, b-1, \ldots, -b$.
Results are in the book's variables $c,\gamma$ (or $a,\alpha$ with `variables="a"`).

In [26]:
def clebsch(b, beta, k, variables="c", show_source=False):
    """One cell of a §8.12 table. Returns the SymPy expression."""
    b, beta, k = Rational(b), Rational(beta), Rational(k)
    for name, val in (("beta", beta), ("k", k)):
        if abs(val) > b or (b - val) % 1:
            raise ValueError(f"{name}={val} must run over {_fmt(b)},...,{_fmt(-b)}")
    expr = cg8.entry(b, beta, k, vars=variables)
    _show(rf"$\langle a\,\alpha,\;{_fmt(b)}\,{_fmt(beta)} \mid "
          rf"{_shift('a', k)},\;\gamma\rangle=$",
          cg8.render(b, beta, k), expr, show_source)
    return expr

In [4]:
# ---- edit these three values and re-run ----
B, BETA, K = Rational(1), Rational(0), Rational(0)

clebsch(B, BETA, K);

**$\langle a\,\alpha,\;1\,0 \mid a,\;\gamma\rangle$**

<IPython.core.display.Math object>

SymPy: `gamma*sqrt(1/(c*(c + 1)))`

In [27]:
if HAS_WIDGETS:
    def _cg_ui():
        b  = W.Dropdown(options=[(_fmt(Rational(i, 2)), Rational(i, 2))
                                 for i in range(1, 11)],       # 1/2 ... 5
                        value=Rational(1), description="b")
        header = W.HTML("<b>  Clebsch-Gordan Coefficient  </b><br>")
        header2=W.Output()
        with header2:
            display(Math(rf"$\langle a\,\alpha,b,\beta \mid c,\gamma\rangle$"))
        k  = W.Dropdown(description="k = c−a")
        be = W.Dropdown(description="β")
        src = W.Checkbox(value=False, description="LaTeX source")

        out = W.Output()
        state = {"busy": False}   # suppress recompute while options are swapped

        def repopulate(*_):
            state["busy"] = True
            opts = [(_fmt(v), v) for v in _vals(b.value)]
            for dd in (k, be):
                keep = dd.value
                dd.options = opts
                dd.value = keep if any(keep == v for _, v in opts) else opts[0][1]
            state["busy"] = False
            recompute()

        def recompute(*_):
            if state["busy"]:
                return
            with out:
                out.clear_output(wait=True)
                clebsch(b.value, be.value, k.value, show_source=src.value)

        b.observe(repopulate, "value")
        for ctl in (k, be, src):
            ctl.observe(recompute, "value")
        repopulate()
        return W.VBox([W.HBox([header, header2]), W.HBox([b, be, k]), src, out])

    display(_cg_ui())
else:
    print("ipywidgets not installed - use the edit-and-rerun cell above.")

In [ ]:
# Whole table. b = 5 is 121 cells and takes a few minutes; start small.
B_TABLE = Rational(1)

_vs = _vals(B_TABLE)
_grid(_vs, _vs,
      lambda k, be: cg8.render(B_TABLE, be, k),
      lambda k: f"$c={_shift('a', k)}$",
      lambda be: rf"$\beta={_fmt(be)}$",
      rf"$b={_fmt(B_TABLE)}$")

---
## §9.11 — $6j$ symbols

$$\begin{Bmatrix} a & b & c \\ d & e & f\end{Bmatrix},
\qquad f = b+m,\quad e = c+n$$

with $a,b,c$ symbolic, in the book's notation
$S=a+b+c$ and $X=-a(a{+}1)+b(b{+}1)+c(c{+}1)$.
Pick the fixed $d$, the **row** $m$ and the **column** $n$.

In [6]:
def sixj(d, m, n, show_source=False):
    """One cell of a §9.11 table. Returns the SymPy expression."""
    d, m, n = Rational(d), Rational(m), Rational(n)
    for name, val in (("m", m), ("n", n)):
        if abs(val) > d or (d - val) % 1:
            raise ValueError(f"{name}={val} must run over {_fmt(d)},...,{_fmt(-d)}")
    expr = sixj9.entry(d, m, n)
    _show(rf"$\begin{{Bmatrix}} a & b & c \\ {_fmt(d)} & {_shift('c', n)} "
          rf"& {_shift('b', m)}\end{{Bmatrix}}$",
          sixj9.render(d, m, n), expr, show_source)
    return expr

In [7]:
# ---- edit these three values and re-run ----
D, M, N = Rational(1), Rational(0), Rational(0)

sixj(D, M, N);

**$\begin{Bmatrix} a & b & c \\ 1 & c & b\end{Bmatrix}$**

<IPython.core.display.Math object>

SymPy: `(-1)**(a + b + c + 1)*sqrt(1/(b*c*(b + 1)*(2*b + 1)*(c + 1)*(2*c + 1)))*(-a**2 - a + b**2 + b + c**2 + c)/2`

In [ ]:
if HAS_WIDGETS:
    def _6j_ui():
        d = W.Dropdown(options=[(_fmt(Rational(i, 2)), Rational(i, 2))
                                for i in range(1, 9)],         # 1/2 ... 4
                       value=Rational(1), description="d")
        m = W.Dropdown(description="m (f=b+m)")
        n = W.Dropdown(description="n (e=c+n)")
        src = W.Checkbox(value=False, description="LaTeX source")
        out = W.Output()
        state = {"busy": False}   # suppress recompute while options are swapped

        def repopulate(*_):
            state["busy"] = True
            opts = [(_fmt(v), v) for v in _vals(d.value)]
            for dd in (m, n):
                keep = dd.value
                dd.options = opts
                dd.value = keep if any(keep == v for _, v in opts) else opts[0][1]
            state["busy"] = False
            recompute()

        def recompute(*_):
            if state["busy"]:
                return
            with out:
                out.clear_output(wait=True)
                sixj(d.value, m.value, n.value, show_source=src.value)

        d.observe(repopulate, "value")
        for ctl in (m, n, src):
            ctl.observe(recompute, "value")
        repopulate()
        return W.VBox([W.HBox([d, m, n]), src, out])

    display(_6j_ui())
else:
    print("ipywidgets not installed - use the edit-and-rerun cell above.")

In [ ]:
# Whole table. d = 4 is 81 cells and is slow; start small.
D_TABLE = Rational(1)

_vs = _vals(D_TABLE)
_grid(_vs, _vs,
      lambda m, n: sixj9.render(D_TABLE, m, n),
      lambda m: f"$f={_shift('b', m)}$",
      lambda n: f"$e={_shift('c', n)}$",
      rf"$d={_fmt(D_TABLE)}$")

---
## §10.11 — $9j$ symbols

$$\begin{Bmatrix} a+\lambda & b+\mu & c+\nu \\ a & b & c \\
\alpha & \beta & \gamma \end{Bmatrix}$$

with $a,b,c$ symbolic, in the book's notation
$S=a+b+c$ and $Z=-c(c{+}1)+a(a{+}1)+b(b{+}1)$.
Pick the table $(\alpha,\beta,\gamma)$, then $\lambda,\mu,\nu$ running over
$\alpha\ldots-\alpha$, $\beta\ldots-\beta$, $\gamma\ldots-\gamma$.

These are the slowest — a few seconds per cell, since each expands into three
$6j$ symbols summed over an intermediate momentum.

In [ ]:
# The (alpha, beta, gamma) of Tables 10.1-10.12.
TABLES_10 = [(Rational(1, 2), Rational(1, 2), 0), (Rational(1, 2), Rational(1, 2), 1),
             (1, 1, 0), (1, 1, 1),
             (Rational(3, 2), Rational(3, 2), 0), (Rational(3, 2), Rational(3, 2), 1),
             (2, 1, 1), (2, 2, 0), (2, 2, 1),
             (Rational(5, 2), Rational(3, 2), 1), (3, 2, 1)]


def ninej(alpha, beta, gamma, lam, mu, nu, show_source=False):
    """One cell of a §10.11 table. Returns the SymPy expression."""
    al, be, ga = Rational(alpha), Rational(beta), Rational(gamma)
    lam, mu, nu = Rational(lam), Rational(mu), Rational(nu)
    for name, val, lim in (("lambda", lam, al), ("mu", mu, be), ("nu", nu, ga)):
        if abs(val) > lim or (lim - val) % 1:
            raise ValueError(f"{name}={val} must run over {_fmt(lim)},...,{_fmt(-lim)}")
    expr = ninej10.entry(al, be, ga, lam, mu, nu)
    _show(rf"$\begin{{Bmatrix}} {_shift('a', lam)} & {_shift('b', mu)} & "
          rf"{_shift('c', nu)} \\ a & b & c \\ {_fmt(al)} & {_fmt(be)} & "
          rf"{_fmt(ga)}\end{{Bmatrix}}$",
          ninej10.render(al, be, ga, lam, mu, nu), expr, show_source)
    return expr

In [ ]:
# ---- edit these and re-run ----
ALPHA, BETA9, GAMMA = Rational(1, 2), Rational(1, 2), Rational(0)
LAM, MU, NU = Rational(1, 2), Rational(1, 2), Rational(0)

ninej(ALPHA, BETA9, GAMMA, LAM, MU, NU);

In [ ]:
if HAS_WIDGETS:
    def _9j_ui():
        tab = W.Dropdown(
            options=[(f"α={_fmt(x)}, β={_fmt(y)}, γ={_fmt(z)}", (x, y, z))
                     for x, y, z in TABLES_10],
            value=TABLES_10[0], description="table")
        lam = W.Dropdown(description="λ")
        mu  = W.Dropdown(description="μ")
        nu  = W.Dropdown(description="ν")
        src = W.Checkbox(value=False, description="LaTeX source")
        go  = W.Button(description="Compute", button_style="primary")
        out = W.Output()

        def repopulate(*_):
            al, be, ga = tab.value
            for dd, lim in ((lam, al), (mu, be), (nu, ga)):
                opts = [(_fmt(v), v) for v in _vals(lim)]
                keep = dd.value
                dd.options = opts
                dd.value = keep if any(keep == v for _, v in opts) else opts[0][1]

        def recompute(*_):
            al, be, ga = tab.value
            with out:
                out.clear_output(wait=True)
                print("computing ...")
                out.clear_output(wait=True)
                ninej(al, be, ga, lam.value, mu.value, nu.value, show_source=src.value)

        tab.observe(repopulate, "value")
        go.on_click(recompute)
        repopulate()
        return W.VBox([tab, W.HBox([lam, mu, nu]), W.HBox([src, go]), out])

    display(_9j_ui())
else:
    print("ipywidgets not installed - use the edit-and-rerun cell above.")

The $9j$ button is explicit rather than auto-updating: each cell takes a few
seconds, so recomputing on every dropdown change would be unpleasant.

In [ ]:
# One (lambda, mu) grid at fixed nu. Slow for the larger tables.
AL9, BE9, GA9, NU9 = Rational(1, 2), Rational(1, 2), Rational(0), Rational(0)

_grid(_vals(AL9), _vals(BE9),
      lambda l, m: ninej10.render(AL9, BE9, GA9, l, m, NU9),
      lambda l: rf"$\lambda={_fmt(l)}$",
      lambda m: rf"$\mu={_fmt(m)}$",
      rf"$\nu={_fmt(NU9)}$")

---
## Verification

Each generator ships with a checker that compares it against SymPy's own
implementation, at two levels where applicable: the Racah engine itself, and
the book-form regrouping (sign, phase, and which factors get absorbed into the
radical). Both levels call the shipped functions, so what is verified is what
is used.

The arguments below cap the range so these run in seconds; omit them for the
full sweep (minutes).

In [ ]:
import subprocess, sys

def check(script, *args):
    r = subprocess.run([sys.executable, str(ROOT / "scripts" / script), *map(str, args)],
                       capture_output=True, text=True, cwd=str(ROOT))
    print(r.stdout or r.stderr)
    return r.returncode

check("check_8_12.py", 3)     # b <= 3/2   (omit arg for b <= 5)
check("check_9_11.py", 2)     # d <= 1     (omit arg for d <= 4)
check("check_10_11.py", 1)    # alpha <= 1/2 (omit arg for alpha <= 3)

### Command line

The same generators run standalone, which is the better route for bulk work:

```bash
python3 scripts/gen_8_12_cg_tables.py  --b 1 --beta=0 --k=0
python3 scripts/gen_9_11_6j_tables.py  --d 1 --m=0 --n=0
python3 scripts/gen_10_11_9j_tables.py --alpha 3/2 --beta 3/2 --gamma 0 --lam=1/2 --mu=1/2 --nu=0
```

Omit the index arguments to dump a whole table. Use `=` for negative values
(`--m=-1`), since a bare leading `-` is read as a flag.

### Caveats

* The **factored arrangement** of a result is a presentational choice. For the
  larger tables the book groups radicals differently (compressing them into
  factorial ratios), so compare numerically rather than character by character.
* Nothing is capped at the book's ranges: `--b 7`, `--d 11/2` all work, just
  more slowly.